In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import numpy as np

# Add src to path just in case
sys.path.append(os.path.abspath(os.getcwd()))

try:
    from src.models.SlotGated import SlotGated
except ImportError:
    # If standard import fails, try adjusting path or direct import
    from src.models.SlotGated import SlotGated

def parse_line(line):
    utterance_data, intent_label = line.split(" <=> ")
    items = utterance_data.split()
    words = [item.rsplit(':', 1)[0] for item in items]
    word_labels = [item.rsplit(':', 1)[1] for item in items]
    return {
        'intent_label': intent_label,
        'words': words,
        'word_labels': word_labels,
        'length': len(words)
    }

def load_data(path):
    lines = Path(path).read_text('utf-8').strip().splitlines()
    return [parse_line(line) for line in lines]

train_data = load_data('dataset/train')
valid_data = load_data('dataset/valid')
test_data = load_data('dataset/test')

print(f"Train size: {len(train_data)}")
print(f"Valid size: {len(valid_data)}")
print(f"Test size: {len(test_data)}")

Train size: 13084
Valid size: 700
Test size: 700


In [2]:
class Vocabulary:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.intent2idx = {}
        self.slot2idx = {'<PAD>': 0}
        
    def build_vocab(self, data):
        for entry in data:
            # Words
            for word in entry['words']:
                if word not in self.word2idx:
                    self.word2idx[word] = len(self.word2idx)
            
            # Intent
            intent = entry['intent_label']
            if intent not in self.intent2idx:
                self.intent2idx[intent] = len(self.intent2idx)
                
            # Slots
            for slot in entry['word_labels']:
                if slot not in self.slot2idx:
                    self.slot2idx[slot] = len(self.slot2idx)
                    
    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary()
vocab.build_vocab(train_data)

print(f"Vocab Size: {len(vocab.word2idx)}")
print(f"Intent Size: {len(vocab.intent2idx)}")
print(f"Slot Size: {len(vocab.slot2idx)}")

Vocab Size: 13457
Intent Size: 7
Slot Size: 73


In [3]:
class IntentSlotDataset(Dataset):
    def __init__(self, data, vocab):
        self.data = data
        self.vocab = vocab
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        entry = self.data[idx]
        
        words = [self.vocab.word2idx.get(w, self.vocab.word2idx['<UNK>']) for w in entry['words']]
        intent = self.vocab.intent2idx.get(entry['intent_label'])
        slots = [self.vocab.slot2idx.get(s, self.vocab.slot2idx['<PAD>']) for s in entry['word_labels']]
        
        return torch.tensor(words), torch.tensor(intent), torch.tensor(slots)

def collate_fn(batch):
    words, intents, slots = zip(*batch)
    lengths = torch.tensor([len(w) for w in words])
    
    # Pad words and slots
    words_padded = torch.nn.utils.rnn.pad_sequence(words, batch_first=True, padding_value=0)
    slots_padded = torch.nn.utils.rnn.pad_sequence(slots, batch_first=True, padding_value=0)
    intents = torch.stack(intents)
    
    return words_padded, lengths, intents, slots_padded

train_dataset = IntentSlotDataset(train_data, vocab)
valid_dataset = IntentSlotDataset(valid_data, vocab)
test_dataset = IntentSlotDataset(test_data, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = SlotGated(
    vocab_size=len(vocab.word2idx),
    embedding_dim=128,
    hidden_dim=256,
    slot_dim=len(vocab.slot2idx),
    intent_dim=len(vocab.intent2idx),
    dropout=0.3
).to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion_intent = nn.CrossEntropyLoss()
criterion_slot = nn.CrossEntropyLoss(ignore_index=0) # Ignore padding

epochs = 10
best_valid_loss = float('inf')

for epoch in range(epochs):
    model.train()
    total_loss = 0
    intent_acc = 0
    total_samples = 0
    
    for words, lengths, intents, slots in train_loader:
        words = words.to(device)
        intents = intents.to(device)
        slots = slots.to(device)
        # lengths usually needs to be on CPU for pack_padded_sequence in older PyTorch versions
        # but modern versions handle it. SlotGated uses .cpu() internally just in case.
        
        optimizer.zero_grad()
        
        intent_logits, slot_logits = model(words, lengths)
        
        loss_intent = criterion_intent(intent_logits, intents)
        loss_slot = criterion_slot(slot_logits.view(-1, len(vocab.slot2idx)), slots.view(-1))
        
        loss = loss_intent + loss_slot
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Simple accuracy tracking for intent
        _, predicted_intent = torch.max(intent_logits, 1)
        intent_acc += (predicted_intent == intents).sum().item()
        total_samples += intents.size(0)
        
    avg_loss = total_loss / len(train_loader)
    train_acc = intent_acc / total_samples
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Train Intent Acc: {train_acc:.4f}")
    
    # Validation
    model.eval()
    valid_loss = 0
    correct_intent = 0
    total_valid = 0
    
    with torch.no_grad():
        for words, lengths, intents, slots in valid_loader:
            words = words.to(device)
            intents = intents.to(device)
            slots = slots.to(device)
            
            intent_logits, slot_logits = model(words, lengths)
            
            loss_intent = criterion_intent(intent_logits, intents)
            loss_slot = criterion_slot(slot_logits.view(-1, len(vocab.slot2idx)), slots.view(-1))
            valid_loss += (loss_intent + loss_slot).item()
            
            _, predicted_intent = torch.max(intent_logits, 1)
            correct_intent += (predicted_intent == intents).sum().item()
            total_valid += intents.size(0)
            
    avg_valid_loss = valid_loss / len(valid_loader)
    valid_acc = correct_intent / total_valid
    print(f"       | Valid Loss: {avg_valid_loss:.4f} | Valid Intent Acc: {valid_acc:.4f}")
    
    if avg_valid_loss < best_valid_loss:
        best_valid_loss = avg_valid_loss
        torch.save(model.state_dict(), 'slot_gated_model.pth')
        print("       | Saved Best Model")

Using device: cuda
Epoch 1 | Loss: 1.4201 | Train Intent Acc: 0.9037
       | Valid Loss: 0.5110 | Valid Intent Acc: 0.9657
       | Saved Best Model
Epoch 2 | Loss: 0.4286 | Train Intent Acc: 0.9776
       | Valid Loss: 0.3009 | Valid Intent Acc: 0.9871
       | Saved Best Model
Epoch 3 | Loss: 0.2721 | Train Intent Acc: 0.9867
       | Valid Loss: 0.2479 | Valid Intent Acc: 0.9814
       | Saved Best Model
Epoch 4 | Loss: 0.1875 | Train Intent Acc: 0.9915
       | Valid Loss: 0.2362 | Valid Intent Acc: 0.9857
       | Saved Best Model
Epoch 5 | Loss: 0.1390 | Train Intent Acc: 0.9946
       | Valid Loss: 0.2402 | Valid Intent Acc: 0.9786
Epoch 6 | Loss: 0.1079 | Train Intent Acc: 0.9961
       | Valid Loss: 0.2085 | Valid Intent Acc: 0.9886
       | Saved Best Model
Epoch 7 | Loss: 0.0825 | Train Intent Acc: 0.9972
       | Valid Loss: 0.2306 | Valid Intent Acc: 0.9829
Epoch 8 | Loss: 0.0642 | Train Intent Acc: 0.9983
       | Valid Loss: 0.2177 | Valid Intent Acc: 0.9871
Epoch 9 | L

In [5]:
from sklearn.metrics import precision_recall_fscore_support, classification_report
from collections import defaultdict

def evaluate_model(model, data_loader, vocab, device):
    """
    Evaluate model with precision, recall, F1 for both intent and slot filling.
    """
    model.eval()
    
    all_intent_preds = []
    all_intent_labels = []
    all_slot_preds = []
    all_slot_labels = []
    
    with torch.no_grad():
        for words, lengths, intents, slots in data_loader:
            words = words.to(device)
            intents = intents.to(device)
            slots = slots.to(device)
            
            intent_logits, slot_logits = model(words, lengths)
            
            # Intent predictions
            intent_preds = torch.argmax(intent_logits, dim=1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_intent_labels.extend(intents.cpu().numpy())
            
            # Slot predictions (flatten, but exclude padding)
            slot_preds = torch.argmax(slot_logits, dim=2)  # [batch, seq_len]
            
            for i in range(slots.size(0)):
                seq_len = lengths[i].item()
                # Only consider non-padded positions
                all_slot_preds.extend(slot_preds[i, :seq_len].cpu().numpy())
                all_slot_labels.extend(slots[i, :seq_len].cpu().numpy())
    
    # Convert to numpy arrays
    all_intent_preds = np.array(all_intent_preds)
    all_intent_labels = np.array(all_intent_labels)
    all_slot_preds = np.array(all_slot_preds)
    all_slot_labels = np.array(all_slot_labels)
    
    # Intent metrics
    intent_precision, intent_recall, intent_f1, _ = precision_recall_fscore_support(
        all_intent_labels, all_intent_preds, average='weighted', zero_division=0
    )
    intent_accuracy = (all_intent_preds == all_intent_labels).mean()
    
    # Slot metrics (excluding PAD token which is index 0)
    # Create mask for non-PAD slots
    non_pad_mask = all_slot_labels != 0
    filtered_slot_labels = all_slot_labels[non_pad_mask]
    filtered_slot_preds = all_slot_preds[non_pad_mask]
    
    slot_precision, slot_recall, slot_f1, _ = precision_recall_fscore_support(
        filtered_slot_labels, filtered_slot_preds, average='weighted', zero_division=0
    )
    slot_accuracy = (filtered_slot_preds == filtered_slot_labels).mean() if len(filtered_slot_labels) > 0 else 0
    
    # Micro and Macro averages for slots
    slot_precision_micro, slot_recall_micro, slot_f1_micro, _ = precision_recall_fscore_support(
        filtered_slot_labels, filtered_slot_preds, average='micro', zero_division=0
    )
    slot_precision_macro, slot_recall_macro, slot_f1_macro, _ = precision_recall_fscore_support(
        filtered_slot_labels, filtered_slot_preds, average='macro', zero_division=0
    )
    
    results = {
        'intent': {
            'accuracy': intent_accuracy,
            'precision': intent_precision,
            'recall': intent_recall,
            'f1': intent_f1
        },
        'slot': {
            'accuracy': slot_accuracy,
            'precision_weighted': slot_precision,
            'recall_weighted': slot_recall,
            'f1_weighted': slot_f1,
            'precision_micro': slot_precision_micro,
            'recall_micro': slot_recall_micro,
            'f1_micro': slot_f1_micro,
            'precision_macro': slot_precision_macro,
            'recall_macro': slot_recall_macro,
            'f1_macro': slot_f1_macro
        }
    }
    
    return results, all_slot_labels, all_slot_preds, all_intent_labels, all_intent_preds

# Evaluate on test set
print("=" * 60)
print("EVALUATION ON TEST SET")
print("=" * 60)

results, slot_labels, slot_preds, intent_labels, intent_preds = evaluate_model(
    model, test_loader, vocab, device
)

print("\n📊 INTENT CLASSIFICATION METRICS:")
print(f"  Accuracy:  {results['intent']['accuracy']:.4f}")
print(f"  Precision: {results['intent']['precision']:.4f}")
print(f"  Recall:    {results['intent']['recall']:.4f}")
print(f"  F1 Score:  {results['intent']['f1']:.4f}")

print("\n📊 SLOT FILLING METRICS:")
print(f"  Accuracy:           {results['slot']['accuracy']:.4f}")
print(f"  Precision (weighted): {results['slot']['precision_weighted']:.4f}")
print(f"  Recall (weighted):    {results['slot']['recall_weighted']:.4f}")
print(f"  F1 Score (weighted):  {results['slot']['f1_weighted']:.4f}")
print(f"  F1 Score (micro):     {results['slot']['f1_micro']:.4f}")
print(f"  F1 Score (macro):     {results['slot']['f1_macro']:.4f}")

# Per-slot-type classification report
print("\n" + "=" * 60)
print("DETAILED SLOT CLASSIFICATION REPORT")
print("=" * 60)

# Get slot names for report
idx2slot = {v: k for k, v in vocab.slot2idx.items()}
slot_names = [idx2slot[i] for i in sorted(vocab.slot2idx.values()) if i != 0]  # Exclude PAD

# Filter to only include slots that appear in labels or predictions
unique_slots = np.union1d(np.unique(slot_labels), np.unique(slot_preds))
unique_slots = unique_slots[unique_slots != 0]  # Remove PAD

target_names = [idx2slot[i] for i in unique_slots]

print(classification_report(
    slot_labels[slot_labels != 0], 
    slot_preds[slot_labels != 0], 
    labels=unique_slots,
    target_names=target_names,
    zero_division=0
))

EVALUATION ON TEST SET

📊 INTENT CLASSIFICATION METRICS:
  Accuracy:  0.9800
  Precision: 0.9808
  Recall:    0.9800
  F1 Score:  0.9800

📊 SLOT FILLING METRICS:
  Accuracy:           0.9550
  Precision (weighted): 0.9590
  Recall (weighted):    0.9550
  F1 Score (weighted):  0.9547
  F1 Score (micro):     0.9550
  F1 Score (macro):     0.8565

DETAILED SLOT CLASSIFICATION REPORT
                              precision    recall  f1-score   support

                           O       0.99      0.98      0.99      3401
               B-entity_name       0.44      0.89      0.59        18
               I-entity_name       0.64      0.91      0.75        54
            B-playlist_owner       0.98      0.94      0.96        54
                  B-playlist       0.89      0.94      0.91       109
                  I-playlist       0.92      0.96      0.94       191
                B-music_item       0.99      0.99      0.99        86
                    B-artist       0.95      0.70      0

In [6]:
def predict(model, sentence, vocab, device):
    model.eval()
    words = sentence.split()
    word_indices = [vocab.word2idx.get(w, vocab.word2idx['<UNK>']) for w in words]
    
    x = torch.tensor(word_indices).unsqueeze(0).to(device) # [1, seq_len]
    lengths = torch.tensor([len(words)])
    
    with torch.no_grad():
        intent_logits, slot_logits = model(x, lengths)
        
    # Decode Intent
    intent_idx = torch.argmax(intent_logits, dim=1).item()
    intent_label = [k for k, v in vocab.intent2idx.items() if v == intent_idx][0]
    
    # Decode Slots
    slot_indices = torch.argmax(slot_logits, dim=2).squeeze(0).tolist()
    slot_labels = []
    idx2slot = {v: k for k, v in vocab.slot2idx.items()}
    
    for idx in slot_indices:
        slot_labels.append(idx2slot.get(idx, '<UNK>'))
        
    return intent_label, list(zip(words, slot_labels))

# Example usage
sample_sentence = "whats the weather like for the next three days?"
print(f"Sentence: {sample_sentence}")

intent, slots = predict(model, sample_sentence, vocab, device)
print(f"Predicted Intent: {intent}")
print("Slots:")
for word, slot in slots:
    print(f"  {word}: {slot}")

Sentence: whats the weather like for the next three days?
Predicted Intent: GetWeather
Slots:
  whats: O
  the: O
  weather: O
  like: O
  for: O
  the: O
  next: B-timeRange
  three: I-timeRange
  days?: I-timeRange
